In [1]:
# pip install elasticsearch

In [2]:
from elasticsearch import Elasticsearch
import warnings
warnings.filterwarnings("ignore")
# Connect to ES with basic authentication
es = Elasticsearch(
 [{'host':'localhost','port':9200, "scheme": "https"}],
 basic_auth=('elastic', 'enVSh0H2=BLqPg9HgpX1'),
 verify_certs=False
)
# Clear existing index to start fresh
if es.indices.exists(index="myindex"):
 es.indices.delete(index='myindex')

In [3]:
path = "./data/booksummaries.txt"
count = 1
for line in open(path, encoding='utf-8'):
 fields = line.split("\t")
 doc = {
 'id' : fields[0],
 'title': fields[2],
 'author': fields[3],
 'summary': fields[6]
 }
 es.index(index="myindex", id=fields[0], body=doc)

 count += 1
 if count == 501: break
print("Indexing complete.")
#Check to see how big is the index
res = es.search(index="myindex", body={"query": {"match_all": {}}})
print("Your index has %d entries" % res['hits']['total']['value'])

Indexing complete.
Your index has 168 entries


In [4]:
#Try a test query. The query searches "summary" field which contains the text
#and does a full text query on that field.
res = es.search(index="myindex", body={"query": {"match": {"summary": "animal"}}})
print("Your search returned %d results." % res['hits']['total']['value'])
#Printing the title field and summary field's first 100 characters for 2nd result
print(res["hits"]["hits"][2]["_source"]["title"])
print(res["hits"]["hits"][2]["_source"]["summary"][:100])
#match query considers both exact matches, and fuzzy matches and works as a OR query.
#match_phrase looks for exact matches.
while True:
 query = input("Enter your search query: ")
 if query == "STOP":
     break
 res = es.search(index="myindex", body={"query": {"match_phrase": {"summary": query}}})
 print("Your search returned %d results:" % res['hits']['total']['value'])
 for hit in res["hits"]["hits"]:
     print(hit["_source"]["title"])
 #to get a snippet 100 characters before and after the match
     loc = hit["_source"]["summary"].lower().index(query)
     print(hit["_source"]["summary"][:100])
     print(hit["_source"]["summary"][loc-100:loc+100])

Your search returned 7 results.
A Streetcar Named Desire
 Blanche DuBois is a fading, but still-attractive, Southern belle whose pretensions to virtue and cu


Enter your search query:  animal


Your search returned 16 results:
Animal Farm
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he co

P.S. Your Cat Is Dead
 Abandoned by his girlfriend on New Year's Eve, and still unaware that his beloved cat Tennessee (na
aware that his beloved cat Tennessee (named after the playwright Tennessee Williams) has died in an animal clinic, hopeless New York actor Jimmy Zoole is feeling depressed and unstable when he happens
Dead Air
 The first person narrative begins on 11 September 2001, and Banks uses the protagonist's conversati
only; makes sense.") and sees him described as a drug and booze fuelled, sexually promiscuous party animal. His politics are left-wing and libertarian, and he rants at every chance. Nott's various gir
The Murders in the Rue Morgue
 The story surrounds the baffling double murder of Madame L'Espanaye and her daughter in the Rue Mor
he sailor reveals that he had been keeping a captive orangutan obtained while ashore in

Enter your search query:  STOP


In [5]:
# pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import nltk 
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('stopwords')
from gensim.corpora import Dictionary
from gensim.models import LdaModel
def preprocess(textstring):
 stops = set(nltk.corpus.stopwords.words('english'))
 tokens = nltk.tokenize.word_tokenize(textstring)
 return [t.lower() for t in tokens if t.isalpha() and t not in stops]
# Build the corpus
summaries = [preprocess(line.split("\t")[6]) for line in 
             open("./data/booksummaries.txt", encoding="utf-8")]
# Create a dictionary representation of the documents.
dictionary = Dictionary(summaries)
# Filter infrequent or too frequent words.
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(s) for s in summaries]
# Make a index to word dictionary.
temp = dictionary[0] # This is only to "load" the dictionary.
id2word = dictionary.id2token

In [9]:
#Train the LAD topic model
model = LdaModel(corpus=corpus, id2word=id2word,iterations=400,
num_topics=10)
top_topics = list(model.top_topics(corpus))
print(top_topics)
for idx in range(10):
 print("Topic #%s:" % idx, model.print_topic(idx, 10))
print("=" * 20)
#Train the LAD topic model
from gensim.models import LsiModel
lsimodel = LsiModel(corpus, num_topics=10, id2word = id2word) # train model
print(lsimodel.print_topics(num_topics=10, num_words=10))
for idx in range(10):
 print("Topic #%s:" % idx, lsamodel.print_topic(idx, 10))
print("=" * 20)

[([(np.float32(0.008766617), 'she'), (np.float32(0.008359596), 'he'), (np.float32(0.006530892), 'tells'), (np.float32(0.005892279), 'one'), (np.float32(0.0056496924), 'back'), (np.float32(0.0055547683), 'they'), (np.float32(0.004824434), 'go'), (np.float32(0.004152602), 'father'), (np.float32(0.0041225287), 'when'), (np.float32(0.004107567), 'house'), (np.float32(0.0038762062), 'find'), (np.float32(0.0038376413), 'time'), (np.float32(0.0037675104), 'get'), (np.float32(0.0036292593), 'day'), (np.float32(0.0036266048), 'after'), (np.float32(0.0035835707), 'home'), (np.float32(0.003296926), 'away'), (np.float32(0.003262105), 'man'), (np.float32(0.0032373164), 'night'), (np.float32(0.003219576), 'mother')], np.float64(-0.9270250382505456)), ([(np.float32(0.0057926397), 'he'), (np.float32(0.005659261), 'she'), (np.float32(0.0056062303), 'father'), (np.float32(0.0048977346), 'mother'), (np.float32(0.0045579905), 'life'), (np.float32(0.004479495), 'one'), (np.float32(0.0041159987), 'love'), (

NameError: name 'lsamodel' is not defined

In [13]:
pip install lxml_html_clean==0.2.2 --force-reinstall

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 64.5 MB/s eta 0:00:00

  Attempting uninstall: lxml

    Found existing installation: lxml 5.3.0

    Uninstalling lxml-5.3.0:

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ------------------------------

In [14]:
pip install sumy

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 8.0/8.0 MB 93.9 MB/s eta 0:00:00
  Created wheel for breadability: filename=breadability-0.1.20-py2.py3-none-any.whl size=21738 sha256=bf3a3374668808f58ff8ee133b6b549b87c93b41ae662bbc54f475c22bf1e84c
  Stored in directory: c:\users\300407626\appdata\local\pip\cache\wheels\56\de\fa\27aad3b8cf9dc6bf1675f414e4dec461a9dac0562fcb439b44
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13775 sha256=a986b40f301e1ddffabdb30580340c20c7c7d3f5fc02361318a89a728f473575
  Stored in directory: c:\users\300407626\appdata\local\pip\cache\wheels\0b\1d\03\175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built breadability docopt

   -------- -----------

  DEPRECATION: Building 'breadability' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'breadability'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'docopt' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'docopt'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [15]:
#Code to summarize a given webpage using Sumy's TextRank implementation.
from sumy.parsers.html import HtmlParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from sumy.summarizers.luhn import LuhnSummarizer
from sumy.summarizers.lsa import LsaSummarizer
num_sentences_in_summary = 2
url = "https://en.wikipedia.org/wiki/Automatic_summarization"
parser = HtmlParser.from_url(url, Tokenizer("english"))
summarizer_list=("TextRankSummarizer:","LexRankSummarizer:","LuhnSummarizer:","LsaSummarizer") #list of summarizers
summarizers = [TextRankSummarizer(), LexRankSummarizer(), LuhnSummarizer(),
LsaSummarizer()]
for i,summarizer in enumerate(summarizers):
 print(summarizer_list[i])
 for sentence in summarizer(parser.document, num_sentences_in_summary):
     print((sentence))
 print("-"*30)

TextRankSummarizer:
For text, extraction is analogous to the process of skimming, where the summary (if available), headings and subheadings, figures, the first and last paragraphs of a section, and optionally the first and last sentences in a paragraph are read before one chooses to read the entire document in detail.
A Class of Submodular Functions for Document Summarization", The 49th Annual Meeting of the Association for Computational Linguistics: Human Language Technologies (ACL-HLT), 2011^ Sebastian Tschiatschek, Rishabh Iyer, Hoachen Wei and Jeff Bilmes, Learning Mixtures of Submodular Functions for Image Collection Summarization, In Advances of Neural Information Processing Systems (NIPS), Montreal, Canada, December - 2014.^ Ramakrishna Bairi, Rishabh Iyer, Ganesh Ramakrishnan and Jeff Bilmes, Summarizing Multi-Document Topic Hierarchies using Submodular Mixtures, To Appear In the Annual Meeting of the Association for Computational Linguistics (ACL), Beijing, China, July - 2015